In [1]:
%pip install --upgrade openai

^C
Note: you may need to restart the kernel to use updated packages.


     ---------------------------------------- 0.0/58.3 kB ? eta -:--:--
     -------------------- ----------------- 30.7/58.3 kB 660.6 kB/s eta 0:00:01
     -------------------- ----------------- 30.7/58.3 kB 660.6 kB/s eta 0:00:01
     -------------------------- ----------- 41.0/58.3 kB 217.9 kB/s eta 0:00:01
     -------------------------------------- 58.3/58.3 kB 255.4 kB/s eta 0:00:00
   ---------------------------------------- 0.0/221.4 kB ? eta -:--:--
   ----------- ---------------------------- 61.4/221.4 kB 1.7 MB/s eta 0:00:01
   ---------------- ----------------------- 92.2/221.4 kB 1.3 MB/s eta 0:00:01
   ---------------- ----------------------- 92.2/221.4 kB 1.3 MB/s eta 0:00:01
   ----------------------------- -------- 174.1/221.4 kB 807.1 kB/s eta 0:00:01
   ------------------------------------ - 215.0/221.4 kB 871.5 kB/s eta 0:00:01
   -------------------------------------- 221.4/221.4 kB 793.9 kB/s eta 0:00:00
   ---------------------------------------- 0.0/75.0 kB ? et

In [16]:
from openai import OpenAI
import json
from dotenv import load_dotenv, find_dotenv

_ : bool = load_dotenv(find_dotenv()) # read local .env file

In [2]:
import json

def show_json(obj):
    display(json.loads(obj.model_dump_json()))

In [4]:
from openai import OpenAI

client = OpenAI()

assistant = client.beta.assistants.create(
    name="Math Tutor",
    instructions="You are a personal math tutor. Answer questions briefly, in a sentence or less.",
    model="gpt-3.5-turbo-1106",
)
show_json(assistant)

{'id': 'asst_UqdUOtjAbVuwszeRIDWDcx23',
 'created_at': 1701607448,
 'description': None,
 'file_ids': [],
 'instructions': 'You are a personal math tutor. Answer questions briefly, in a sentence or less.',
 'metadata': {},
 'model': 'gpt-3.5-turbo-1106',
 'name': 'Math Tutor',
 'object': 'assistant',
 'tools': []}

In [5]:
thread = client.beta.threads.create()
show_json(thread)

{'id': 'thread_P7MSX7oe6dD7JNm8gpbxAO2r',
 'created_at': 1701607481,
 'metadata': {},
 'object': 'thread'}

In [6]:
message = client.beta.threads.messages.create(
    thread_id=thread.id,
    role="user",
    content="I want you to solve this equation using PEDMAS: 3x4/5**7+45",
)
show_json(message)

{'id': 'msg_QllyW7CjQmLGAgZTacaRnFyB',
 'assistant_id': None,
 'content': [{'text': {'annotations': [],
    'value': 'I want you to solve this equation using PEDMAS: 3x4/5**7+45'},
   'type': 'text'}],
 'created_at': 1701607590,
 'file_ids': [],
 'metadata': {},
 'object': 'thread.message',
 'role': 'user',
 'run_id': None,
 'thread_id': 'thread_P7MSX7oe6dD7JNm8gpbxAO2r'}

In [7]:
run = client.beta.threads.runs.create(
    thread_id=thread.id,
    assistant_id=assistant.id,
)
show_json(run)

{'id': 'run_6GE5TFCbmENniUppcGv4vbHI',
 'assistant_id': 'asst_UqdUOtjAbVuwszeRIDWDcx23',
 'cancelled_at': None,
 'completed_at': None,
 'created_at': 1701607606,
 'expires_at': 1701608206,
 'failed_at': None,
 'file_ids': [],
 'instructions': 'You are a personal math tutor. Answer questions briefly, in a sentence or less.',
 'last_error': None,
 'metadata': {},
 'model': 'gpt-3.5-turbo-1106',
 'object': 'thread.run',
 'required_action': None,
 'started_at': None,
 'status': 'queued',
 'thread_id': 'thread_P7MSX7oe6dD7JNm8gpbxAO2r',
 'tools': []}

In [8]:
import time

def wait_on_run(run, thread):
    while run.status == "queued" or run.status == "in_progress":
        run = client.beta.threads.runs.retrieve(
            thread_id=thread.id,
            run_id=run.id,
        )
        time.sleep(0.5)
    return run

In [21]:
run = wait_on_run(run, thread)
show_json(run)

{'id': 'run_6GE5TFCbmENniUppcGv4vbHI',
 'assistant_id': 'asst_UqdUOtjAbVuwszeRIDWDcx23',
 'cancelled_at': None,
 'completed_at': None,
 'created_at': 1701607606,
 'expires_at': None,
 'failed_at': 1701607612,
 'file_ids': [],
 'instructions': 'You are a personal math tutor. Answer questions briefly, in a sentence or less.',
 'last_error': {'code': 'rate_limit_exceeded',
  'message': 'You exceeded your current quota, please check your plan and billing details.'},
 'metadata': {},
 'model': 'gpt-3.5-turbo-1106',
 'object': 'thread.run',
 'required_action': None,
 'started_at': 1701607607,
 'status': 'failed',
 'thread_id': 'thread_P7MSX7oe6dD7JNm8gpbxAO2r',
 'tools': []}

In [17]:
messages = client.beta.threads.messages.list(thread_id=thread.id)
show_json(messages)

{'data': [{'id': 'msg_QllyW7CjQmLGAgZTacaRnFyB',
   'assistant_id': None,
   'content': [{'text': {'annotations': [],
      'value': 'I want you to solve this equation using PEDMAS: 3x4/5**7+45'},
     'type': 'text'}],
   'created_at': 1701607590,
   'file_ids': [],
   'metadata': {},
   'object': 'thread.message',
   'role': 'user',
   'run_id': None,
   'thread_id': 'thread_P7MSX7oe6dD7JNm8gpbxAO2r'}],
 'object': 'list',
 'first_id': 'msg_QllyW7CjQmLGAgZTacaRnFyB',
 'last_id': 'msg_QllyW7CjQmLGAgZTacaRnFyB',
 'has_more': False}